# Tags and Vote Activity

Explore current tag composition and historical monthly vote activity by tag. The tag table is a current snapshot; it is not a historical question-count panel.

In [ ]:
from pathlib import Path
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt

def root():
    for p in [Path.cwd(),*Path.cwd().parents]:
        q=p/'stack_exchange_analysis'
        if (p/'database').exists(): return p
        if (q/'database').exists(): return q
    raise FileNotFoundError
PROJECT_ROOT=root(); DATA_DIR=PROJECT_ROOT/'database'; ANALYSIS_DIR=PROJECT_ROOT/'analysis'; ANALYSIS_DIR.mkdir(exist_ok=True); sys.path.insert(0,str(PROJECT_ROOT/'src'))
from analysis_utils import read_csv_flexible, save_figure

## Current tag snapshot

In [ ]:
tags=read_csv_flexible(DATA_DIR/'tags-db.csv'); tag_col=next((c for c in tags if c.lower() in {'tagname','tag'}),None); count_col=next((c for c in tags if c.lower()=='count'),None)
if tag_col and count_col:
    tags[count_col]=pd.to_numeric(tags[count_col],errors='coerce'); top=tags.nlargest(30,count_col)[[tag_col,count_col]]; display(top)
    fig,ax=plt.subplots(figsize=(9,8)); ax.barh(top[tag_col][::-1],top[count_col][::-1]); ax.set(title='Largest tags in current snapshot',xlabel='Tag count'); ax.grid(axis='x',alpha=.2); save_figure(fig,ANALYSIS_DIR/'tags_top_snapshot.png'); plt.show()
else: display(tags.head())

## Historical monthly vote activity by tag

In [ ]:
votes=read_csv_flexible(DATA_DIR/'History-Sum-By-Month-Per-Year-of-Votes-by-Tag.csv'); name_col=next((c for c in votes if c.lower()=='tagname'),None); month_col=next((c for c in votes if 'month' in c.lower()),None)
if name_col is None or month_col is None: raise ValueError(f'Expected TagName and month column; got {list(votes.columns)}')
votes[month_col]=pd.to_datetime(votes[month_col],errors='coerce')
for c in ['Upvotes','Downvotes','TotalVotes','NetVotes','UpvoteDownvoteRatio']:
    if c in votes: votes[c]=pd.to_numeric(votes[c],errors='coerce')
votes=votes.dropna(subset=[month_col,name_col]); print(votes[month_col].min(),votes[month_col].max())

## Technology trajectories

In [ ]:
value='TotalVotes' if 'TotalVotes' in votes else 'Upvotes'; p=votes.pivot_table(index=month_col,columns=name_col,values=value,aggfunc='sum').sort_index(); selected=[t for t in ['python','javascript','java','c#','c++','sql','r','php','html','css'] if t in p]
fig,ax=plt.subplots(figsize=(12,5)); [ax.plot(p.index,p[t].rolling(3,min_periods=1).mean(),label=t) for t in selected]; ax.set(title=f'Monthly {value} by selected tag',xlabel='Month',ylabel=value); ax.legend(ncol=5,fontsize=8); ax.grid(alpha=.2); save_figure(fig,ANALYSIS_DIR/'tag_vote_trajectories.png'); plt.show()

## Concentration and entropy

In [ ]:
m=votes.pivot_table(index=month_col,columns=name_col,values=value,aggfunc='sum',fill_value=0).sort_index(); shares=m.div(m.sum(axis=1).replace(0,np.nan),axis=0); hhi=(shares**2).sum(axis=1); entropy=-(shares.where(shares>0)*np.log(shares.where(shares>0))).sum(axis=1); composition=pd.DataFrame({'HHI':hhi,'Entropy':entropy}); display(composition.tail())
fig,ax=plt.subplots(figsize=(12,4)); ax.plot(composition.index,composition.HHI); ax.set(title='Concentration of vote activity across tags',xlabel='Month',ylabel='HHI'); ax.grid(alpha=.2); save_figure(fig,ANALYSIS_DIR/'tag_vote_hhi.png'); plt.show()

## Takeaways
This dataset supports analysis of technological attention and voting composition. Direct analysis of new-question composition still requires a historical questions-by-tag export.